# Motion Generation: Guide a Body Through Poses

Motion generation (rigid body guidance) is the most constrained form of
four-bar synthesis. Unlike path generation which only prescribes *where* a
coupler point should go, motion generation prescribes both **position and
orientation** of a rigid body attached to the coupler link.

Applications include:
- Pick-and-place mechanisms that must reorient a part
- Hatch and door opening mechanisms
- Robot end-effector positioning
- Assembly line part-flipping devices

**What you'll learn:**
- Defining poses (position + orientation) for a rigid body
- Synthesizing a four-bar linkage that guides the body through those poses
- Classifying solutions with the Grashof criterion
- Converting solutions to simulatable linkages
- Validating the mechanism against the target poses
- Constraining ground pivot locations

In [ ]:
import math

import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
from matplotlib.patches import Rectangle

from pylinkage.synthesis import Pose, motion_generation, solution_to_linkage
from pylinkage.synthesis.utils import grashof_check
from pylinkage.visualizer import plot_static_linkage

## 1. Define the Target Poses

A `Pose` consists of an (x, y) position and an orientation angle (in radians).
We define three poses representing a part being translated and rotated,
as if being flipped from a horizontal pickup to an angled placement.

In [ ]:
poses = [
    Pose(0, 0, 0),                # Start: origin, no rotation
    Pose(2, 1, math.pi / 6),      # Mid: translated right and up, rotated 30 deg
    Pose(3, -0.5, math.pi / 3),   # End: further right, down, rotated 60 deg
]

for i, p in enumerate(poses):
    print(f"Pose {i + 1}: x={p.x:.2f}, y={p.y:.2f}, angle={math.degrees(p.angle):.1f} deg")

In [ ]:
# Visualize the target poses as rectangles with orientation arrows
fig, ax = plt.subplots(figsize=(10, 6))

rect_w, rect_h = 1.0, 0.4  # Body dimensions for visualization
colors = ['#2196F3', '#FF9800', '#4CAF50']

for i, (pose, color) in enumerate(zip(poses, colors, strict=False)):
    # Draw a rotated rectangle at each pose
    rect = Rectangle(
        (-rect_w / 2, -rect_h / 2), rect_w, rect_h,
        linewidth=2, edgecolor=color, facecolor=color, alpha=0.3,
    )
    t = (
        mtransforms.Affine2D()
        .rotate(pose.angle)
        .translate(pose.x, pose.y)
        + ax.transData
    )
    rect.set_transform(t)
    ax.add_patch(rect)

    # Orientation arrow
    dx = 0.6 * math.cos(pose.angle)
    dy = 0.6 * math.sin(pose.angle)
    ax.annotate(
        '', xy=(pose.x + dx, pose.y + dy), xytext=(pose.x, pose.y),
        arrowprops=dict(arrowstyle='->', color=color, lw=2),
    )
    ax.plot(pose.x, pose.y, 'o', color=color, markersize=8)
    ax.annotate(
        f'Pose {i + 1}\n({pose.x}, {pose.y}, {math.degrees(pose.angle):.0f}$^\\circ$)',
        (pose.x, pose.y), textcoords='offset points', xytext=(10, 10),
        fontsize=9, color=color,
    )

ax.set_xlim(-1.5, 5)
ax.set_ylim(-2, 3)
ax.set_aspect('equal')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title('Target Poses for Motion Generation')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Synthesize the Four-Bar Linkage

With 3 poses, Burmester theory yields a continuous curve of solutions.
The `motion_generation` function samples this curve and returns up to
`max_solutions` valid four-bar linkages.

In [ ]:
result = motion_generation(
    poses=poses,
    max_solutions=10,
    require_grashof=True,
)

print(f"Found {len(result.raw_solutions)} solutions")
if result.warnings:
    for w in result.warnings:
        print(f"  Warning: {w}")

## 3. Classify Solutions with Grashof Criterion

The Grashof criterion tells us whether any link can make a full rotation.
A **crank-rocker** is typically the most useful: the crank rotates fully
while the rocker oscillates.

In [ ]:
for i, sol in enumerate(result.raw_solutions):
    grashof_type = grashof_check(
        sol.crank_length, sol.coupler_length,
        sol.rocker_length, sol.ground_length,
    )
    print(
        f"Solution {i}: "
        f"crank={sol.crank_length:.3f}, coupler={sol.coupler_length:.3f}, "
        f"rocker={sol.rocker_length:.3f}, ground={sol.ground_length:.3f}  "
        f"-> {grashof_type.name}"
    )

## 4. Convert to Linkage and Simulate

Pick the solution with the best pose accuracy, convert it to a `Linkage`
object, and run a full simulation cycle.

> **Note:** Burmester synthesis guarantees the coupler body passes through
> the target poses at specific crank angles. During continuous simulation,
> however, the circle-circle intersection solver may pick the wrong
> assembly mode at intermediate angles, causing small deviations from the
> target poses. This is a well-known limitation of four-bar simulation.

In [ ]:
if not result.raw_solutions:
    raise RuntimeError(
        "No solutions found. Try require_grashof=False or different poses."
    )

# Evaluate each solution's pose accuracy and pick the best one
def evaluate_pose_accuracy(sol, poses, iterations=360):
    """Return max distance from coupler point to any target pose."""
    lkg = solution_to_linkage(sol, iterations=iterations)
    from pylinkage.exceptions import UnbuildableError
    try:
        loci = tuple(zip(*lkg.step(), strict=False))
    except UnbuildableError:
        return float("inf")  # can't assemble
    names = [j.name for j in lkg.components]
    p_idx = names.index('P') if 'P' in names else names.index('C')

    max_dist = 0
    for pose in poses:
        best_d = float('inf')
        for step in range(len(loci[p_idx])):
            pos = loci[p_idx][step]
            if pos[0] is None:
                continue
            d = math.sqrt((pos[0] - pose.x) ** 2 + (pos[1] - pose.y) ** 2)
            best_d = min(best_d, d)
        max_dist = max(max_dist, best_d)
    return max_dist

best_idx = 0
best_accuracy = float('inf')
for i, sol in enumerate(result.raw_solutions):
    acc = evaluate_pose_accuracy(sol, poses)
    if acc < best_accuracy:
        best_accuracy = acc
        best_idx = i

best_sol = result.raw_solutions[best_idx]
linkage = solution_to_linkage(best_sol, name="motion_gen", iterations=120)

grashof_type = grashof_check(
    best_sol.crank_length, best_sol.coupler_length,
    best_sol.rocker_length, best_sol.ground_length,
)
print(f"Using solution {best_idx} ({grashof_type.name}, max pose error: {best_accuracy:.4f})")
print(f"Linkage: {linkage.name}")
print(f"Joints: {[j.name for j in linkage.components]}")
print(f"Ground pivots: A={best_sol.ground_pivot_a}, D={best_sol.ground_pivot_d}")

In [ ]:
# Run simulation and collect all joint positions
loci = tuple(zip(*linkage.step(), strict=False))

print(f"Simulated {len(loci[0])} steps, tracking {len(loci)} joints")

In [ ]:
# Plot the mechanism motion with ghost frames
fig, ax = plt.subplots(figsize=(10, 8))

# Convert loci to list-of-tuples format for plot_static_linkage
loci_frames = list(zip(*loci, strict=False))

# Draw mechanism with ghost outlines using plot_static_linkage
plot_static_linkage(
    linkage, ax, loci_frames,
    show_legend=True, show_labels=True, show_loci=True,
    n_ghosts=6,
    title="Synthesized Four-Bar Mechanism Motion",
)

# Draw coupler point P locus highlighted
joint_names = [j.name for j in linkage.components]
if 'P' in joint_names:
    p_idx = joint_names.index('P')
    px = [pos[0] for pos in loci[p_idx] if pos[0] is not None]
    py = [pos[1] for pos in loci[p_idx] if pos[1] is not None]
    if px:
        ax.plot(
            px, py, '-', linewidth=1.5, color='purple',
            alpha=0.6, label='Coupler point P locus',
        )

# Mark target poses
for i, pose in enumerate(poses):
    ax.plot(pose.x, pose.y, 'r*', markersize=14)
    ax.annotate(f'Pose {i + 1}', (pose.x, pose.y),
                textcoords='offset points', xytext=(8, 8), fontsize=9, color='red')

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5. Validation: Overlay Target Poses with Mechanism Poses

The key validation step: we draw the coupler body at each precision pose
and overlay the target poses to verify alignment. The coupler body is
represented by the triangle B-C-P (crank pivot, coupler pivot, coupler point).

We compute where the mechanism's coupler body actually ends up at each
simulation step and find the frames closest to the target poses.

In [ ]:
def find_closest_frame(loci, joint_names, target_pose):
    """Find the simulation frame where the coupler point is closest to target."""
    # If there's a coupler point P, use it; otherwise use joint C
    p_idx = joint_names.index('P') if 'P' in joint_names else joint_names.index('C')
    n_steps = len(loci[p_idx])

    best_dist = float('inf')
    best_frame = 0

    for step in range(n_steps):
        pos = loci[p_idx][step]
        if pos[0] is None or pos[1] is None:
            continue
        dist = math.sqrt((pos[0] - target_pose.x) ** 2 + (pos[1] - target_pose.y) ** 2)
        if dist < best_dist:
            best_dist = dist
            best_frame = step

    return best_frame, best_dist


def get_coupler_angle(loci, joint_names, frame):
    """Compute the coupler orientation at a given frame from joint B and C."""
    b_idx = joint_names.index('B')
    c_idx = joint_names.index('C')
    bx, by = loci[b_idx][frame]
    cx, cy = loci[c_idx][frame]
    return math.atan2(cy - by, cx - bx)


# Find the closest simulation frame for each target pose
print("Pose matching results:")
print(f"{'Pose':<8} {'Target (x,y,angle)':<30} {'Closest dist':<15} {'Frame'}")

matched_frames = []
for i, pose in enumerate(poses):
    frame, dist = find_closest_frame(loci, joint_names, pose)
    matched_frames.append(frame)
    angle_at_frame = get_coupler_angle(loci, joint_names, frame)
    print(
        f"  {i + 1:<6} ({pose.x:.2f}, {pose.y:.2f}, {math.degrees(pose.angle):.1f} deg)"
        f"   {dist:<15.4f} {frame}"
    )

In [ ]:
# Validation plot: overlay target poses with mechanism state at matched frames
fig, ax = plt.subplots(figsize=(12, 8))

colors_pose = ['#2196F3', '#FF9800', '#4CAF50']

# Convert loci to frame-major format and draw mechanism with ghost outlines
loci_frames = list(zip(*loci, strict=False))
plot_static_linkage(
    linkage, ax, loci_frames,
    show_legend=False, show_labels=False, show_loci=True,
    n_ghosts=0,
)

# Draw ground pivot labels
ax.annotate('A', best_sol.ground_pivot_a, textcoords='offset points',
            xytext=(-12, -12), fontsize=10, fontweight='bold')
ax.annotate('D', best_sol.ground_pivot_d, textcoords='offset points',
            xytext=(-12, -12), fontsize=10, fontweight='bold')

b_idx = joint_names.index('B')
c_idx = joint_names.index('C')

rect_w, rect_h = 1.0, 0.4

for i, (pose, frame, color) in enumerate(zip(poses, matched_frames, colors_pose, strict=False)):
    # Draw target pose as a dashed rectangle
    rect_target = Rectangle(
        (-rect_w / 2, -rect_h / 2), rect_w, rect_h,
        linewidth=2.5, edgecolor=color, facecolor='none',
        linestyle='--', label=f'Target pose {i + 1}',
    )
    t_target = (
        mtransforms.Affine2D()
        .rotate(pose.angle)
        .translate(pose.x, pose.y)
        + ax.transData
    )
    rect_target.set_transform(t_target)
    ax.add_patch(rect_target)

    # Draw mechanism state at matched frame
    bx, by = loci[b_idx][frame]
    cx, cy = loci[c_idx][frame]

    if bx is not None and cx is not None:
        # Draw the linkage bars at this frame
        ax.plot(
            [best_sol.ground_pivot_a[0], bx], [best_sol.ground_pivot_a[1], by],
            '-o', color=color, linewidth=2, markersize=5, alpha=0.7,
        )
        ax.plot(
            [bx, cx], [by, cy],
            '-o', color=color, linewidth=2, markersize=5, alpha=0.7,
        )
        ax.plot(
            [cx, best_sol.ground_pivot_d[0]], [cy, best_sol.ground_pivot_d[1]],
            '-o', color=color, linewidth=2, markersize=5, alpha=0.7,
        )

        # Draw actual coupler body orientation (solid rectangle)
        actual_angle = math.atan2(cy - by, cx - bx)
        rect_actual = Rectangle(
            (-rect_w / 2, -rect_h / 2), rect_w, rect_h,
            linewidth=2, edgecolor=color, facecolor=color, alpha=0.15,
            label=f'Actual pose {i + 1}' if i == 0 else None,
        )
        # If coupler point P exists, center on it; otherwise center on B-C midpoint
        if 'P' in joint_names:
            p_idx = joint_names.index('P')
            px, py = loci[p_idx][frame]
            center_x, center_y = px, py
        else:
            midx, midy = (bx + cx) / 2, (by + cy) / 2
            center_x, center_y = midx, midy
        t_actual = (
            mtransforms.Affine2D()
            .rotate(actual_angle)
            .translate(center_x, center_y)
            + ax.transData
        )
        rect_actual.set_transform(t_actual)
        ax.add_patch(rect_actual)

    # Target orientation arrow
    arrow_len = 0.6
    ax.annotate(
        '', xy=(pose.x + arrow_len * math.cos(pose.angle),
                pose.y + arrow_len * math.sin(pose.angle)),
        xytext=(pose.x, pose.y),
        arrowprops=dict(arrowstyle='->', color=color, lw=2.5, linestyle='--'),
    )

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title('Validation: Target Poses (dashed) vs Mechanism Poses (solid)')
ax.legend(fontsize=8, loc='best')
plt.tight_layout()
plt.show()

## 6. Multiple Solutions Comparison

If multiple solutions were found, compare them side by side.

In [ ]:
n_solutions = min(len(result.raw_solutions), 4)

if n_solutions > 1:
    fig, axes = plt.subplots(1, n_solutions, figsize=(5 * n_solutions, 5))
    if n_solutions == 1:
        axes = [axes]

    for idx, (sol, ax) in enumerate(zip(result.raw_solutions[:n_solutions], axes, strict=False)):
        lkg = solution_to_linkage(sol, name=f"sol_{idx}", iterations=120)
        sol_loci = list(lkg.step())

        # Draw mechanism using plot_static_linkage
        grashof_type = grashof_check(
            sol.crank_length, sol.coupler_length,
            sol.rocker_length, sol.ground_length,
        )
        plot_static_linkage(
            lkg, ax, sol_loci,
            show_legend=False, show_labels=False, show_loci=True,
            title=f'Solution {idx}\n{grashof_type.name}',
        )

        for pose in poses:
            ax.plot(pose.x, pose.y, 'r*', markersize=12)

    plt.suptitle('Comparison of Synthesis Solutions', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print(f"Only {n_solutions} solution(s) found, skipping comparison plot.")

## 7. Synthesis with Fixed Ground Pivots

In many practical designs, the ground pivot locations are constrained by
the machine frame. We can specify them explicitly and let the synthesis
find a compatible linkage.

In [ ]:
# Specify ground pivot locations
ground_a = (-1.0, -1.0)
ground_d = (4.0, -1.0)

result_fixed = motion_generation(
    poses=poses,
    ground_pivot_a=ground_a,
    ground_pivot_d=ground_d,
    max_solutions=10,
    require_grashof=False,  # Relax Grashof to increase chance of finding solutions
)

print(f"Found {len(result_fixed.raw_solutions)} solutions with fixed ground pivots")
if result_fixed.warnings:
    for w in result_fixed.warnings:
        print(f"  Warning: {w}")

In [ ]:
if result_fixed.raw_solutions:
    sol_fixed = result_fixed.raw_solutions[0]
    lkg_fixed = solution_to_linkage(sol_fixed, name="fixed_ground", iterations=120)
    loci_fixed = tuple(zip(*lkg_fixed.step(), strict=False))
    names_fixed = [j.name for j in lkg_fixed.components]

    fig, ax = plt.subplots(figsize=(10, 8))

    for j_idx, name in enumerate(names_fixed):
        xs = [p[0] for p in loci_fixed[j_idx] if p[0] is not None]
        ys = [p[1] for p in loci_fixed[j_idx] if p[1] is not None]
        if xs:
            ax.plot(xs, ys, '-', linewidth=1.2, label=f'Joint {name}')

    ax.plot(*ground_a, 'ks', markersize=10)
    ax.plot(*ground_d, 'ks', markersize=10)
    ax.annotate('A (fixed)', ground_a, textcoords='offset points',
                xytext=(-15, -15), fontsize=9)
    ax.annotate('D (fixed)', ground_d, textcoords='offset points',
                xytext=(5, -15), fontsize=9)

    for i, (pose, color) in enumerate(zip(poses, colors_pose, strict=False)):
        ax.plot(pose.x, pose.y, '*', color=color, markersize=14)
        dx = 0.5 * math.cos(pose.angle)
        dy = 0.5 * math.sin(pose.angle)
        ax.annotate(
            '', xy=(pose.x + dx, pose.y + dy), xytext=(pose.x, pose.y),
            arrowprops=dict(arrowstyle='->', color=color, lw=2),
        )
        ax.annotate(f'Pose {i + 1}', (pose.x, pose.y),
                    textcoords='offset points', xytext=(8, 8),
                    fontsize=9, color=color)

    grashof_type = grashof_check(
        sol_fixed.crank_length, sol_fixed.coupler_length,
        sol_fixed.rocker_length, sol_fixed.ground_length,
    )
    ax.set_title(
        f'Motion Generation with Fixed Ground Pivots ({grashof_type.name})'
    )
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_aspect('equal')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No solutions found with these ground pivots. Try different locations.")

## Summary

Motion generation synthesizes a four-bar linkage whose coupler guides a rigid
body through specified poses (position + orientation). Key takeaways:

| Aspect | Detail |
|---|---|
| **Input** | List of `Pose(x, y, angle)` objects (3-5 poses) |
| **Method** | Burmester theory (circle point / center point curves) |
| **3 poses** | Continuous curve of solutions |
| **4 poses** | Up to 6 discrete solutions |
| **5 poses** | Typically 0-2 solutions (over-constrained) |
| **Filtering** | Grashof criterion, link ratio limits, assembly validation |
| **Output** | `SynthesisResult` with `raw_solutions` and `Linkage` objects |

**Tips for practical use:**
- Start with 3 poses for the most solutions, add more to constrain further
- Set `require_grashof=False` if no solutions are found
- Use `ground_pivot_a` / `ground_pivot_d` when the frame geometry is constrained
- Always validate by overlaying target poses on the simulated mechanism
- The coupler point P (when present) tracks the body reference origin through the poses